In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMRegressor
import gradio as gr

In [ ]:
data = pd.read_csv("./DATA/insurance_data.csv")
data.head()

In [ ]:
data.dropna(axis=0,inplace=True)
data.head()

In [ ]:
features = ['age','gender','bmi','bloodpressure','diabetic','children','smoker','region']
target = 'claim'
categorical_cols = ['gender','diabetic','smoker','region']
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  data[col] = le.fit_transform(data[col])
  label_encoders[col] = le
X = data[features]
y = data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LGBMRegressor(random_state=42)
model.fit(X_train, y_train)

In [ ]:
def predict_claims(csv_file):
  input_data = pd.read_csv(csv_file.name)
  patient_ids = input_data['PatientID']
  for col in categorical_cols:
    le = label_encoders[col]
    input_data[col] = input_data[col].map(lambda x: le.transform([x])[0] if x in le.classes_ else 0)
  X_input = input_data[features]
  predictions = model.predict(X_input)
  results = pd.DataFrame({
    'PatientID': patient_ids,
    'predictedClaims': predictions
  })
  return results

In [ ]:
iface = gr.Interface(
  fn = predict_claims,
  inputs = gr.File(file_types=[".csv"]),
  outputs = "dataframe",
  title = "Insurance Claim Prediction (LightGBM)",
  description = "Upload CSV file with PatientID, age, gender, bmi, blood pressure, diabetic status, amount of children, smoking status, region"
)
iface.launch()